# Causal discovery on Atari frames

Jacqueline Maasch | March 2026

## Preamble

In [1]:
# General imports.
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

# Time series causal discovery.
import tigramite
from tigramite import data_processing as pp
from tigramite.toymodels import structural_causal_processes as toys
from tigramite import plotting as tp
from tigramite.pcmci import PCMCI # also for PCMCI+
from tigramite.independence_tests.gsquared import Gsquared # univariate discrete/categorical vars
from tigramite.independence_tests.cmisymb import CMIsymb # multivariate discrete/categorical vars (permutation-based)

# Vanilla causal discovery.
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.cit import CIT

# Graphical modeling.
import networkx as nx

/Users/jmaasch/anaconda3/envs/icp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define functions

In [2]:
def plot_nx(adjacency_matrix,
            labels,
            figsize = (5,5),
            dpi = 75,
            node_size = 800,
            arrow_size = 10):
    g = nx.from_numpy_array(adjacency_matrix, create_using = nx.DiGraph)
    plt.figure(figsize = figsize, dpi = dpi)  
    nx.draw_circular(g, 
                     node_size = node_size, 
                     node_color = "pink",
                     labels = dict(zip(list(range(len(labels))), labels)), 
                     arrowsize = arrow_size,
                     with_labels = True)
    plt.show()
    plt.close()


def get_ci_test_oracle_fn(graph_true: nx.Graph):
    graph_true = nx.relabel.relabel_nodes(graph_true, {l: i for i, l in enumerate(graph_true.nodes)})
    def ci_test_oracle(values: np.array, x: int, y: int, covar: tuple[int]) -> float:
        p_val = 1 if nx.d_separated(graph_true, {x}, {y}, covar) else 0
        return p_val
    return ci_test_oracle

## Read data

In [3]:
# For the Venture game.
# https://ale.farama.org/environments/venture/
file = "simulate_trajectory/data_atari_games_wr/data__Venture__10000.pkl"
with open(file, "rb") as f:
    trajectory_dict = pickle.load(f)

## Process data

In [4]:
len(trajectory_dict.keys())

10000

In [5]:
len(trajectory_dict[10])

100

In [6]:
trajectory_dict[0][0]

(6,
 {'lives': 4,
  'episode_frame_number': 1,
  'frame_number': 1,
  'labels': {'sprite0_y': 5,
   'sprite1_y': 25,
   'sprite2_y': 50,
   'sprite3_y': 50,
   'sprite4_y': 20,
   'sprite5_y': 50,
   'sprite0_x': 11,
   'sprite1_x': 61,
   'sprite2_x': 61,
   'sprite3_x': 1,
   'sprite4_x': 111,
   'sprite5_x': 151,
   'player_x': 68,
   'player_y': 76,
   'current_room': 8,
   'num_lives': 3,
   'score_1_2': 0,
   'score_3_4': 0}},
 0.0)

In [7]:
trajectory_dict[0][0][1]

{'lives': 4,
 'episode_frame_number': 1,
 'frame_number': 1,
 'labels': {'sprite0_y': 5,
  'sprite1_y': 25,
  'sprite2_y': 50,
  'sprite3_y': 50,
  'sprite4_y': 20,
  'sprite5_y': 50,
  'sprite0_x': 11,
  'sprite1_x': 61,
  'sprite2_x': 61,
  'sprite3_x': 1,
  'sprite4_x': 111,
  'sprite5_x': 151,
  'player_x': 68,
  'player_y': 76,
  'current_room': 8,
  'num_lives': 3,
  'score_1_2': 0,
  'score_3_4': 0}}

In [8]:
# Get relevant features.
step_features = list(trajectory_dict[0][0][1])
step_features.remove("labels")
step_features.remove("episode_frame_number") # episode_frame_number equiv. to frame_number
frame_features = list(trajectory_dict[0][0][1]["labels"].keys())
print(step_features)
print(frame_features)

['lives', 'frame_number']
['sprite0_y', 'sprite1_y', 'sprite2_y', 'sprite3_y', 'sprite4_y', 'sprite5_y', 'sprite0_x', 'sprite1_x', 'sprite2_x', 'sprite3_x', 'sprite4_x', 'sprite5_x', 'player_x', 'player_y', 'current_room', 'num_lives', 'score_1_2', 'score_3_4']


In [9]:
features = step_features + frame_features
print(len(features))
print(features)

20
['lives', 'frame_number', 'sprite0_y', 'sprite1_y', 'sprite2_y', 'sprite3_y', 'sprite4_y', 'sprite5_y', 'sprite0_x', 'sprite1_x', 'sprite2_x', 'sprite3_x', 'sprite4_x', 'sprite5_x', 'player_x', 'player_y', 'current_room', 'num_lives', 'score_1_2', 'score_3_4']


In [10]:
# Init processed data dict.
processed_dict = dict(zip(features,[[]]*len(features)))
processed_dict["experiment_id"] = []
processed_dict["action"] = []
processed_dict["reward"] = []
processed_dict

{'lives': [],
 'frame_number': [],
 'sprite0_y': [],
 'sprite1_y': [],
 'sprite2_y': [],
 'sprite3_y': [],
 'sprite4_y': [],
 'sprite5_y': [],
 'sprite0_x': [],
 'sprite1_x': [],
 'sprite2_x': [],
 'sprite3_x': [],
 'sprite4_x': [],
 'sprite5_x': [],
 'player_x': [],
 'player_y': [],
 'current_room': [],
 'num_lives': [],
 'score_1_2': [],
 'score_3_4': [],
 'experiment_id': [],
 'action': [],
 'reward': []}

In [ ]:
# Key = experiment ID, value = trajectory data.
N_EPISODES = 1 #10k is max
for exp_id,traj in trajectory_dict.items():
    print(len(np.unique(processed_dict["experiment_id"])))
    while len(np.unique(processed_dict["experiment_id"])) <= N_EPISODES:
        for step in traj:
            action,data,reward = step[0],step[1],step[2]
            processed_dict["experiment_id"].append(exp_id)
            processed_dict["action"].append(action)
            processed_dict["reward"].append(reward)
            for f in step_features:
                processed_dict[f] = processed_dict[f]+[data[f]]
            for f in frame_features:
                processed_dict[f]= processed_dict[f]+[data["labels"][f]]
    else:
        break

0


In [ ]:
len(np.unique(processed_dict["experiment_id"]))

In [ ]:
#for key,val in processed_dict.items():
#    print(key)
#    print(len(val))

In [ ]:
df = pd.DataFrame(processed_dict)
df

In [ ]:
df.reward.unique()

In [ ]:
df.action.unique()

In [ ]:
df.frame_number.unique()

## Vanilla PC

In [ ]:
# Chi-Square independence test.
# https://causal-learn.readthedocs.io/en/latest/independence_tests_index/chisq.html
alpha = 0.01
g_pred = pc(df.to_numpy(), indep_test = "chisq", alpha = alpha)

# Visualization using pydot.
g_pred.draw_pydot_graph()

# Save the graph.
pyd = GraphUtils.to_pydot(g_pred.G, labels = list(df.columns))
#pyd.write_png(f"pc_venture.png")

# Convert to networkx.
a_pred = g_pred.G.graph.copy()
a_pred = np.triu(a_pred)
a_pred[a_pred == -1] = 1
g_pred_nx = nx.from_numpy_array(a_pred, create_using = nx.DiGraph)

In [ ]:
#a_pred

In [ ]:
plot_nx(a_pred,
        labels = list(df.columns),
        figsize = (5,5),
        dpi = 75,
        node_size = 800,
        arrow_size = 10)

In [ ]:
# G-Square independence test.
# https://causal-learn.readthedocs.io/en/latest/independence_tests_index/gsq.html
alpha = 0.01
g_pred = pc(df.to_numpy(), indep_test = "gsq", alpha = alpha)

# Visualization using pydot.
g_pred.draw_pydot_graph()

# Save the graph.
pyd = GraphUtils.to_pydot(g_pred.G, labels = list(df.columns))
#pyd.write_png(f"pc_venture.png")

# Convert to networkx.
a_pred = g_pred.G.graph.copy()
a_pred = np.triu(a_pred)
a_pred[a_pred == -1] = 1
g_pred_nx = nx.from_numpy_array(a_pred, create_using = nx.DiGraph)

In [ ]:
plot_nx(a_pred,
        labels = list(df.columns),
        figsize = (5,5),
        dpi = 75,
        node_size = 800,
        arrow_size = 10)

## Time-series methods

### PCMCI+

https://github.com/jakobrunge/tigramite/blob/master/tutorials/causal_discovery/tigramite_tutorial_pcmciplus.ipynb

#### Demo

In [ ]:
#tp.plot_timeseries?

In [ ]:
# Demo dataframe construction.
seed = 7
auto_coeff = 0.95
coeff = 0.4
T = 500
def lin(x): return x

links ={0: [((0, -1), auto_coeff, lin),
            ((1, -1), coeff, lin)
            ],
        1: [((1, -1), auto_coeff, lin), 
            ],
        2: [((2, -1), auto_coeff, lin), 
            ((3, 0), -coeff, lin), 
            ],
        3: [((3, -1), auto_coeff, lin), 
            ((1, -2), coeff, lin), 
            ],
        4: [((4, -1), auto_coeff, lin), 
            ((3, 0), coeff, lin), 
            ],   
        5: [((5, -1), 0.5*auto_coeff, lin), 
            ((6, 0), coeff, lin), 
            ],  
        6: [((6, -1), 0.5*auto_coeff, lin), 
            ((5, -1), -coeff, lin), 
            ],  
        7: [((7, -1), auto_coeff, lin), 
            ((8, 0), -coeff, lin), 
            ],  
        8: [],                                     
        }

# Specify dynamical noise term distributions, here unit variance Gaussians
random_state = np.random.RandomState(seed)
noises = noises = [random_state.randn for j in links.keys()]
    
data, nonstationarity_indicator = toys.structural_causal_process(
    links=links, T=T, noises=noises, seed=seed)
T, N = data.shape

# Initialize dataframe object, specify variable names
var_names = [r'$X^{%d}$' % j for j in range(N) ]
dataframe = pp.DataFrame(data, var_names=var_names)

In [ ]:
dataframe?

In [ ]:
tp.plot_timeseries(dataframe, figsize=(15, 5)); plt.show()

#### Atari data

In [ ]:
# Config.
TAU_MAX = 10
N_EPISODES = 10e3 #10k is max
ALPHA = 0.01

In [ ]:
df

In [ ]:
# Process data.
# Key = experiment ID, value = trajectory data.
steps = 100
episode_arrays = []
episode_dfs = []
max_ids = 1
experiment_ids = []
for exp_id,traj in trajectory_dict.items():
    while len(np.unique(experiment_ids)) <= max_ids:
        experiment_ids.append(exp_id)
        traj_dict = dict(zip(features,[[]]*len(features)))
        traj_dict["experiment_id"] = []
        traj_dict["action"] = []
        traj_dict["reward"] = []
        for step in traj:
            action,data,reward = step[0],step[1],step[2]
            traj_dict["experiment_id"].append(exp_id)
            traj_dict["action"].append(action)
            traj_dict["reward"].append(reward)
            for f in step_features:
                traj_dict[f] = traj_dict[f]+[data[f]]
            for f in frame_features:
                traj_dict[f]= traj_dict[f]+[data["labels"][f]]
        df_traj = pd.DataFrame(traj_dict)
        episode_dfs.append(df_traj)
        episode_arrays.append(df_traj.to_numpy())

In [ ]:
# Sanity check.
print(episode_arrays[0].shape)
print(len(episode_arrays))
display(episode_arrays[:2])

In [ ]:
# Test for expected shape: episodes M, steps T, vars N.
'''
# from docs for tigramite.data_processing.DataFrame
if analysis_mode == 'multiple':
     Numpy array of shape (multiple datasets M, observations T,
     variables N)
     OR
     Dictionary whose values are numpy arrays of shape
     (observations T_i, variables N), where the number of observations
     T_i may vary across the multiple datasets but the number of variables
     N is fixed. 
'''
episode_arrays = np.array(episode_arrays)
episode_arrays.shape

In [ ]:
# Set dtype to int.
print(episode_arrays.dtype)
print(episode_arrays.dtype.itemsize)
episode_arrays = episode_arrays.astype("int16")
print(episode_arrays.dtype)
print(episode_arrays.dtype.itemsize)

In [ ]:
# Cast as tigramite dataframe.
df_tg = pp.DataFrame(episode_arrays, 
                     var_names = df.columns,
                     analysis_mode = "multiple")
df_tg

In [ ]:
# Sanity check.
print(df_tg.var_names)
print("Total episodes:", df_tg.M)
print("Time steps per trajectory:", df_tg.T[0])
print("Total variables per trajectory:", df_tg.N)

In [ ]:
tp.plot_timeseries(df_tg, figsize=(20, 20)); plt.show()

In [ ]:
# Init independence test and discovery algorithm.
gsq = Gsquared(significance = "analytic")
pcmci = PCMCI(dataframe = df_tg, 
              cond_ind_test = gsq,
              verbosity = 1)

In [ ]:
#pcmci.run_bivci?
PCMCI?

In [ ]:
#df_tg.values

In [ ]:
%%capture
'''
# This does not seem to work, possibly due to multiple datasets / shape.
#
#
# Run bivariate, lagged conditional independence test (similar to bivariate Granger causality, but lag-specific). 
# This can help to identify which maximal time lag tau_max to choose.
# Another option would be to plot get_lagged_dependencies, but large autocorrelation will inflate lag peaks.
# run_bivci at least conditions out some part of the autocorrelation.
correlations = pcmci.run_bivci(tau_max = 10, val_only = True)["val_matrix"]
lag_func_matrix = tp.plot_lagfuncs(val_matrix = correlations, 
                                   setup_args = {'var_names':var_names, 
                                                 'figsize':(15, 10), 
                                                 'x_base':5, 
                                                 'y_base':.5})
'''

In [ ]:
%%capture
'''
# This does not seem to work, possibly due to multiple datasets / shape.
#
#
matrix_lags = np.argmax(np.abs(correlations), axis = 2)
tp.plot_densityplots(dataframe = df_tg, 
                     setup_args = {"figsize":(15, 10)}, 
                     add_densityplot_args = {"matrix_lags":matrix_lags})
plt.show()
'''

In [ ]:
# Run PCMCI+.
pcmci.verbosity = 2
results = pcmci.run_pcmciplus(tau_min = 0, 
                              tau_max = TAU_MAX, 
                              pc_alpha = ALPHA)

#### View Atari results

In [ ]:
print("Graph")
print (results['graph'])
print("Adjacency MCI partial correlations")
print (results['val_matrix'].round(2))
print("Adjacency p-values")
print (results['p_matrix'].round(3))

In [ ]:
'''
As mentioned above, p_matrix and val_matrix for PCMCIplus quantify the uncertainty and strength, 
respectively, only for the adjacencies in phase 2, but not for the directionality of contemporaneous 
links determined in phases 3 and 4. Since adjacency is a symmetric property for contemporaneous links, 
p_matrix and val_matrix are symmetric for tau=0, unlike the graph. 
We can correct the p-values by False Discovery Rate (FDR) control yielding the q_matrix.
'''
q_matrix = pcmci.get_corrected_pvalues(p_matrix = results["p_matrix"], 
                                       fdr_method = "fdr_bh",
                                       exclude_contemporaneous = False)
q_matrix

In [ ]:
# View process graph.
tp.plot_graph(val_matrix = results["val_matrix"],
              graph = results["graph"],
              var_names = df.columns,
              link_colorbar_label = "cross-MCI (edges)",
              node_colorbar_label = "auto-MCI (nodes)")
plt.show()

In [ ]:
# Plot time series graph.
tp.plot_time_series_graph(figsize = (8, 8),
                          node_size = 0.05,
                          val_matrix = results["val_matrix"],
                          graph = results["graph"],
                          var_names = df.columns,
                          link_colorbar_label = "MCI")
plt.show()

# End document